In [2]:
import shutil
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder

In [3]:
PDF_PATH = "../data"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL = "gemma4:31b-cloud"
CHROMA_DIR = "./chroma_db"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 4

In [4]:
docs = DirectoryLoader(PDF_PATH, glob="**/*.pdf", loader_cls=PyPDFLoader).load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(docs)
print(f"Loaded {len(docs)} pages, split into {len(chunks)} chunks.")

Loaded 289 pages, split into 157 chunks.


In [5]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
shutil.rmtree(CHROMA_DIR, ignore_errors=True)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
llm = ChatOllama(model=LLM_MODEL, temperature=0)

prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer the question using only the context below. "
    "If the context does not contain the answer, say you don't know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

chain = prompt | llm | StrOutputParser()

In [7]:
reranker = CrossEncoder(RERANK_MODEL)

def expand(q):
    out = llm.invoke(
        "Fix the spelling of this question, then write 2 alternative phrasings "
        "using synonyms. Return one per line, nothing else.\n\nQuestion: " + q
    ).content
    return [l.strip() for l in out.split("\n") if l.strip()][:3] or [q]

def rerank(q, docs):
    scores = reranker.predict([(q, d.page_content) for d in docs])
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [d for _, d in ranked[:TOP_K]]

def retrieve(q):
    variants = expand(q)
    candidates = {}
    for v in variants:
        for d in vectorstore.similarity_search(v, k=6):
            candidates[d.page_content] = d
    return variants[0], rerank(variants[0], list(candidates.values()))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [8]:
history = []

def ask(question):
    search_q = question
    if history:
        past = "\n".join(f"User: {q}\nAI: {a}" for q, a in history[-3:])
        search_q = llm.invoke(
            f"Chat history:\n{past}\n\n"
            f"Rewrite this new question so it makes sense on its own. "
            f"Return only the question.\n\nNew question: {question}"
        ).content

    clean_q, found = retrieve(search_q)
    answer = chain.invoke({"context": format_docs(found), "question": clean_q})
    pages = sorted({(Path(d.metadata["source"]).name, d.metadata["page"] + 1) for d in found})

    history.append((question, answer))
    return answer, pages

In [9]:
while True:
    question = input("\nAsk a question (or 'quit'): ").strip()
    if question.lower() in {"quit", "exit", "q"}:
        break
    if not question:
        continue
    print("\nMe :", question)
    answer, pages = ask(question)
    print("\nGemma:", answer)
    print("\nSources:")
    for name, page in pages:
        print(f"  {name}, page {page}")


Me : hi

Gemma: I don't know.

Sources:
  MLF Lecture 1.pdf, page 43
  MLF Lecture 3.pdf, page 26
  MLF Lecture 3.pdf, page 31
  MLF Lecture 3.pdf, page 37

Me : what is linear regression

Gemma: Linear Regression is a supervised learning algorithm used to predict a continuous numerical value.

Sources:
  MLF Lecture 1.pdf, page 19
  MLF Lecture 5.pdf, page 3
  MLF Lecture 5.pdf, page 4
  MLF Lecture 5.pdf, page 5
